# Retail Sales Data Analysis

## Final Capstone Project

This project analyses retail sales transaction data from four South African store branches.

My goal is to clean the dataset, calculate revenue, analyse sales performance, create visualisations, and use the results to make practical business recommendations.

The dataset is synthetic and was created for the Melsoft Academy final capstone project.

## 1. Import Libraries

I will begin by importing the libraries needed for the analysis.

- Pandas will be used to load, clean and analyse the sales data.
- Matplotlib will be used later to create charts and visualise the results.

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the Dataset

I will now load the original sales data from the data folder into a Pandas DataFrame.

I am keeping the original CSV unchanged because all data cleaning must be performed in Python.

In [7]:
df = pd.read_csv("data/sales_data.csv")

In [8]:
df.head()

,date,transaction_id,product_id,product_name,category,store,quantity,unit_price,payment_method
0,2026-01-31,T1000,P006,Full Cream Milk 2L,Dairy,Sandton,9.0,34.0,Cash
1,2026-06-03,T1001,P010,Beef Mince 1kg,Meat,Pretoria,9.0,135.0,Mobile
2,2026-05-31,T1002,P008,Tomatoes 1kg,Produce,Sandton,4.0,30.0,Card
3,2026-01-29,T1003,P005,Cheddar Cheese 500g,Dairy,Sandton,14.0,95.0,Mobile
4,2026-01-03,T1004,P008,Tomatoes 1kg,Produce,Soweto,11.0,30.0,Cash


In [9]:
df.shape

(600, 9)

## 3. Initial Data Inspection

Before cleaning the dataset, I will inspect its structure, data types, missing values, duplicate rows and inconsistent values.

This helps me understand the quality of the raw data before making any changes.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            600 non-null    object 
 1   transaction_id  600 non-null    object 
 2   product_id      600 non-null    object 
 3   product_name    600 non-null    object 
 4   category        600 non-null    object 
 5   store           600 non-null    object 
 6   quantity        599 non-null    float64
 7   unit_price      600 non-null    object 
 8   payment_method  600 non-null    object 
dtypes: float64(1), object(8)
memory usage: 42.3+ KB


In [11]:
df.isnull().sum()

date              0
transaction_id    0
product_id        0
product_name      0
category          0
store             0
quantity          1
unit_price        0
payment_method    0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(1)

In [13]:
df["category"].unique()

array(['Dairy', 'Meat', 'Produce', 'Beverages', 'Bakery', 'beverages'],
      dtype=object)

In [14]:
df["store"].unique()

array(['Sandton', 'Pretoria', 'Soweto', 'Rosebank', '  soweto '],
      dtype=object)

### Specific Data-Quality Checks

The initial inspection showed missing data, a duplicate row and inconsistent text values.

I will now investigate the remaining known data-quality problems before cleaning the dataset.

In [15]:
pd.to_numeric(df["unit_price"], errors="coerce").isnull().sum()

np.int64(1)

In [16]:
df[pd.to_numeric(df["unit_price"], errors="coerce").isnull()]

,date,transaction_id,product_id,product_name,category,store,quantity,unit_price,payment_method
42,2026-01-12,T1042,P009,Chicken Breasts 1kg,Meat,Sandton,15.0,unknown,Mobile


In [17]:
df[df["quantity"] < 0]

,date,transaction_id,product_id,product_name,category,store,quantity,unit_price,payment_method
200,2026-02-12,T1200,P009,Chicken Breasts 1kg,Meat,Soweto,-3.0,120.0,Mobile


In [18]:
standard_dates = pd.to_datetime(
    df["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df[standard_dates.isnull()]

,date,transaction_id,product_id,product_name,category,store,quantity,unit_price,payment_method
120,03/15/2026,T1120,P003,Whole Wheat Bread,Bakery,Soweto,1.0,28.0,EFT


## 4. Data Cleaning

I identified several data-quality problems in the raw dataset.

I will now clean a copy of the original DataFrame so that the raw data remains unchanged.

The cleaning process will handle missing quantities, non-numeric prices, negative quantities, duplicate rows, inconsistent date formats and inconsistent text values.

In [34]:
df_clean = df.copy()

starting_rows = len(df_clean)

print("Starting rows:", starting_rows)

Starting rows: 600


### Missing Quantity

A transaction with no quantity cannot be used reliably because revenue depends on the number of units sold.

I will count the missing quantity values and remove those rows.

In [35]:
missing_quantity = df_clean["quantity"].isnull().sum()

print("Missing quantity rows:", missing_quantity)

df_clean = df_clean.dropna(subset=["quantity"])

Missing quantity rows: 1


In [36]:
df_clean.shape

(599, 9)

### Invalid Unit Price

The unit_price column should contain numeric values, but the raw data contains a non-numeric value.

I will convert the column to numeric values. Any value that cannot be converted will become missing and will then be removed.

In [37]:
df_clean["unit_price"] = pd.to_numeric(
    df_clean["unit_price"],
    errors="coerce"
)

invalid_price = df_clean["unit_price"].isnull().sum()

print("Invalid unit price rows:", invalid_price)

df_clean = df_clean.dropna(subset=["unit_price"])

Invalid unit price rows: 1


In [38]:
df_clean.shape

(598, 9)

### Negative Quantity

A normal sales transaction cannot contain a negative quantity.

I will identify and remove any rows where the quantity is below zero.

In [39]:
negative_quantity = (df_clean["quantity"] < 0).sum()

print("Negative quantity rows:", negative_quantity)

df_clean = df_clean[df_clean["quantity"] >= 0]

Negative quantity rows: 1


In [40]:
df_clean.shape

(597, 9)

### Duplicate Transactions

Duplicate rows can cause sales and revenue to be counted more than once.

I will count and remove exact duplicate rows.

In [41]:
duplicate_rows = df_clean.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

df_clean = df_clean.drop_duplicates()

Duplicate rows: 1


In [42]:
df_clean.shape

(596, 9)

### Date Formatting

The dataset contains a date that uses a different format from the others.

I will convert the entire date column into Pandas datetime values so that all dates are interpreted correctly before I perform any monthly analysis.

In [43]:
df_clean["date"] = pd.to_datetime(
    df_clean["date"],
    format="mixed",
    errors="coerce"
)

print("Invalid dates after conversion:", df_clean["date"].isnull().sum())

Invalid dates after conversion: 0


In [44]:
df_clean["date"].head()

0   2026-01-31
1   2026-06-03
2   2026-05-31
3   2026-01-29
4   2026-01-03
Name: date, dtype: datetime64[ns]

### Inconsistent Text Values

Some text values contain inconsistent capitalisation or extra spaces.

For example, "beverages" should be grouped with "Beverages", while "  soweto " should be grouped with "Soweto".

I will remove unnecessary spaces and standardise the capitalisation of the relevant text columns.

In [45]:
text_columns = ["category", "store", "payment_method"]

for column in text_columns:
    df_clean[column] = df_clean[column].str.strip().str.title()

In [46]:
print("Categories:")
print(df_clean["category"].unique())

print("\nStores:")
print(df_clean["store"].unique())

print("\nPayment methods:")
print(df_clean["payment_method"].unique())

Categories:
['Dairy' 'Meat' 'Produce' 'Beverages' 'Bakery']

Stores:
['Sandton' 'Pretoria' 'Soweto' 'Rosebank']

Payment methods:
['Cash' 'Mobile' 'Card' 'Eft']


### Cleaning Verification

I will perform final checks to confirm that the major data-quality problems have been removed or corrected successfully.

In [47]:
print("Clean dataset shape:", df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Negative quantities:", (df_clean["quantity"] < 0).sum())

Clean dataset shape: (596, 9)
Missing values: 0
Duplicate rows: 0
Negative quantities: 0


### Data Quality Report

I will summarise the cleaning process so that there is a clear record of the problems found, the rows removed and the final number of usable transactions.

In [48]:
final_rows = len(df_clean)
total_removed = starting_rows - final_rows

print("=" * 40)
print("DATA QUALITY REPORT")
print("=" * 40)

print(f"Starting rows:              {starting_rows}")
print(f"Missing quantity removed:   {missing_quantity}")
print(f"Invalid price removed:      {invalid_price}")
print(f"Negative quantity removed:  {negative_quantity}")
print(f"Duplicate rows removed:     {duplicate_rows}")
print("-" * 40)
print(f"Total rows removed:         {total_removed}")
print(f"Clean rows remaining:       {final_rows}")

print("=" * 40)

DATA QUALITY REPORT
Starting rows:              600
Missing quantity removed:   1
Invalid price removed:      1
Negative quantity removed:  1
Duplicate rows removed:     1
----------------------------------------
Total rows removed:         4
Clean rows remaining:       596


### Cleaning Summary

I started with 600 sales transactions. I removed four unusable rows: one with a missing quantity, one with an invalid unit price, one with a negative quantity and one exact duplicate.

I also corrected the inconsistent date format and standardised inconsistent text values instead of removing those transactions.

After cleaning, I was left with 596 valid transactions that can now be used for the sales analysis.